In [1]:
import sys, json
sys.path.append("..")
%load_ext autoreload
%autoreload 2

import torch
from torch.utils.data import DataLoader
from src.tokenizer import CharTokenizer
from src.dataset import SQLDataset, collate, IGNORE

def load(name):
    return [json.loads(line) for line in open(f"../data/raw/{name}.jsonl", encoding="utf-8")]

train, val, test = load("train"), load("val"), load("test")
tok = CharTokenizer.build([r["question"] for r in train] + [r["sql"] for r in train])

ds = SQLDataset(train, tok)
x, y = ds[0]
print("x:", x.shape, x.dtype, "| y:", y.shape)
print("masked targets:", (y == IGNORE).sum().item(), "| real targets:", (y != IGNORE).sum().item())

x: torch.Size([104]) torch.int64 | y: torch.Size([104])
masked targets: 41 | real targets: 63


In [2]:
# The real targets, decoded, must equal the SQL (decode drops <eos>)
targets = [t for t in y.tolist() if t != IGNORE]
assert tok.decode(targets) == train[0]["sql"]
print("targets decode to the SQL: OK")

# The input is the whole text, minus the last token
print(tok.decode(x.tolist()))

# One batch of 4
loader = DataLoader(ds, batch_size=4, shuffle=True, collate_fn=lambda b: collate(b, tok.pad_id))
xb, yb = next(iter(loader))
print("batch shapes:", xb.shape, yb.shape)
print("pad ids in x:", (xb == tok.pad_id).sum().item(), "| ignored in y:", (yb == IGNORE).sum().item())

targets decode to the SQL: OK
average price in the electronics categorySELECT AVG(price) FROM products WHERE category = 'electronics'
batch shapes: torch.Size([4, 205]) torch.Size([4, 205])
pad ids in x: 229 | ignored in y: 365


In [1]:
   import sys; print(sys.executable)

C:\Users\makar\Python\tiny-transformer-lab\.venv\Scripts\python.exe
